# Retrievers

A **retriever** is a LangChain **Runnable** that accepts a **query string** and returns **`list[Document]`** — the chunks most relevant to that query. Vector stores (**09. Embeddings and Vector Stores**) implement search internally; **`.as_retriever()`** wraps them with a standard interface so RAG chains can plug retrieval into LCEL without caring which database backs the index.

This notebook covers the retriever contract, **`VectorStoreRetriever`** options (`similarity`, **MMR**, score thresholds), metadata filters, custom retriever implementations, composing retrievers in **`RunnableParallel`**, batch invocation, testing with fake retrievers, and patterns that extend basic search (multi-query, compression, ensemble — conceptual previews before **11. RAG with LCEL**).

Prerequisites: **08. Documents and Text Splitting**, **09. Embeddings and Vector Stores**, **02. Runnable Protocol and LCEL**, **06. LCEL Composition Patterns**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import uuid

import langchain_core
import numpy as np

np.set_printoptions(precision=4, suppress=True)

print("langchain-core:", langchain_core.__version__)

---

## 1. What Is a Retriever?

```
user query (str) ──► Retriever.invoke() ──► list[Document]
```

| Property | Detail |
|----------|--------|
| **Input** | Query string (natural language question) |
| **Output** | Ordered list of **`Document`** chunks |
| **Interface** | **`Runnable`** — also supports `batch`, `ainvoke`, etc. |
| **Hidden work** | Embed query, search vector store, optional rerank/filter |

RAG chains treat the retriever as a black box:

```python
RunnableParallel(
    context=retriever | format_docs,
    question=RunnablePassthrough(),
)
| rag_prompt | llm | StrOutputParser()
```

The retriever step is where **recall** happens — if the right chunk is not retrieved, the LLM cannot answer correctly.

---

## 2. Build the Indexed Vector Store

Standalone setup — same lesson corpus as **09**:

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

CORPUS = [
    Document(page_content="The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE.", metadata={"id": "c1", "source": "api-docs"}),
    Document(page_content="Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions.", metadata={"id": "c2", "source": "db-docs"}),
    Document(page_content="JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header.", metadata={"id": "c3", "source": "security-docs"}),
    Document(page_content="Pytest fixtures share database setup. Use conftest.py for session-scoped engines.", metadata={"id": "c4", "source": "test-docs"}),
    Document(page_content="Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files.", metadata={"id": "c5", "source": "db-docs"}),
]

vectorstore = Chroma.from_documents(
    documents=CORPUS,
    embedding=embeddings,
    collection_name=f"retriever_lesson_{uuid.uuid4().hex[:8]}",
)

print("indexed:", vectorstore._collection.count(), "documents")

---

## 3. VectorStoreRetriever — as_retriever()

**`.as_retriever()`** converts a vector store into a **`VectorStoreRetriever`**:

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

docs = retriever.invoke("How does JWT authentication work?")
print(type(retriever).__name__)
print(f"returned {len(docs)} documents:")
for d in docs:
    print(f"  [{d.metadata['id']}] {d.page_content[:65]}...")

### 3.1 Key Parameters

| Parameter | Values | Meaning |
|-----------|--------|--------|
| **`search_type`** | `"similarity"`, `"mmr"`, `"similarity_score_threshold"` | Search algorithm |
| **`search_kwargs.k`** | int | Number of documents to return |
| **`search_kwargs.fetch_k`** | int (MMR) | Candidates before diversity filter |
| **`search_kwargs.lambda_mult`** | 0–1 (MMR) | Relevance vs diversity trade-off |
| **`search_kwargs.filter`** | dict | Metadata filter passed to vector store |
| **`search_kwargs.score_threshold`** | float | Minimum similarity (threshold mode) |

---

## 4. Similarity Search — Default

**`search_type="similarity"`** returns the **k** nearest vectors by embedding distance — pure relevance ranking.

In [ ]:
sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

for question in [
    "Alembic migration command",
    "pytest conftest fixtures",
    "FastAPI notes routes",
]:
    hits = sim_retriever.invoke(question)
    ids = [d.metadata["id"] for d in hits]
    print(f"Q: {question}")
    print(f"  ids: {ids}\n")

---

## 5. MMR — Maximal Marginal Relevance

**MMR** reduces **redundant** chunks (e.g. two nearly identical Alembic paragraphs) by balancing relevance with diversity.

$$\text{MMR} = \arg\max_{D_i \in R \setminus S}\Big[ \lambda \cdot \text{sim}(Q, D_i) - (1-\lambda) \cdot \max_{D_j \in S} \text{sim}(D_i, D_j) \Big]$$

- **`fetch_k`**: pool of candidates from initial similarity search
- **`lambda_mult`**: 1 = pure relevance, 0 = pure diversity

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5},
)

sim_hits = sim_retriever.invoke("Alembic schema migrations")
mmr_hits = mmr_retriever.invoke("Alembic schema migrations")

print("similarity ids:", [d.metadata["id"] for d in sim_hits])
print("MMR ids:       ", [d.metadata["id"] for d in mmr_hits])

Use **MMR** when retrieved chunks often overlap (duplicate policy sections, repeated boilerplate). Use **similarity** when chunks are already distinct.

---

## 6. Score Threshold — Filter Weak Matches

**`search_type="similarity_score_threshold"`** returns only documents above a similarity threshold — helps avoid injecting irrelevant context.

In [ ]:
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 5},
)

for q in ["JWT bearer header", "completely unrelated quantum physics"]:
    try:
        hits = threshold_retriever.invoke(q)
        print(f"Q: {q}")
        print(f"  hits: {len(hits)} {[d.metadata['id'] for d in hits]}\n")
    except Exception as e:
        print(f"Q: {q} → no docs above threshold ({e})\n")

Tune **`score_threshold`** on a labeled query set. When zero docs pass, RAG prompts should instruct the model to say "I don't know" (**11. RAG with LCEL**).

---

## 7. Metadata Filters on Retrievers

Pass **`filter`** in **`search_kwargs`** to scope retrieval:

In [ ]:
security_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2, "filter": {"source": "security-docs"}},
)

db_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2, "filter": {"source": "db-docs"}},
)

print("security filter:", [d.metadata["id"] for d in security_retriever.invoke("authentication")])
print("db filter:      ", [d.metadata["id"] for d in db_retriever.invoke("migrations")])

Combine with **routers** (**06**) — classify intent, then invoke the retriever with the appropriate metadata filter.

---

## 8. format_docs — From Documents to Prompt Text

Retrievers return **`Document`** objects; prompts need a **string**. Standard helper:

In [ ]:
def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "(no relevant context found)"
    return "\n\n".join(
        f"[{d.metadata.get('id', '?')}] {d.page_content}" for d in docs
    )


sample_docs = retriever.invoke("JWT header")
print(format_docs(sample_docs))

Use in LCEL: **`retriever | format_docs`** or wrap **`format_docs`** inside **`RunnableLambda`** when the retriever output is nested in a dict.

---

## 9. Custom Retriever — RunnableLambda

Any function **`str → list[Document]`** can serve as a retriever when wrapped in **`RunnableLambda`**:

In [ ]:
from langchain_core.runnables import RunnableLambda

KEYWORD_INDEX = {
    "jwt": [CORPUS[2]],
    "auth": [CORPUS[2]],
    "alembic": [CORPUS[1], CORPUS[4]],
    "pytest": [CORPUS[3]],
    "fastapi": [CORPUS[0]],
}


def keyword_retrieve(query: str) -> list[Document]:
    q = query.lower()
    seen = set()
    results = []
    for key, docs in KEYWORD_INDEX.items():
        if key in q:
            for d in docs:
                did = d.metadata["id"]
                if did not in seen:
                    seen.add(did)
                    results.append(d)
    return results or [CORPUS[0]]


custom_retriever = RunnableLambda(keyword_retrieve)
print(custom_retriever.invoke("JWT bearer token"))
print("Runnable:", hasattr(custom_retriever, "batch"))

---

## 10. Custom Retriever — BaseRetriever

For classes with config and LangChain serialization, subclass **`BaseRetriever`**:

In [ ]:
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.retrievers import BaseRetriever
from pydantic import Field


class TopKVectorRetriever(BaseRetriever):
    """Wrapper that delegates to a vector store with fixed k."""

    vectorstore: Chroma
    k: int = Field(default=2)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> list[Document]:
        return self.vectorstore.similarity_search(query, k=self.k)


topk = TopKVectorRetriever(vectorstore=vectorstore, k=2)
print(topk.invoke("database migrations"))

`BaseRetriever` integrates with **callbacks** (**15. Callbacks, Caching, and Observability**) — `run_manager` propagates tracing context.

---

## 11. Retriever in RAG LCEL (Preview)

Full RAG assembly is **11. RAG with LCEL** — the retrieval slot uses the retriever directly:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. If insufficient, say you don't know.\n\n{context}"),
    ("human", "{question}"),
])

rag_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

rag_chain = (
    RunnableParallel(
        context=rag_retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What header carries the JWT token?"))

---

## 12. batch and async on Retrievers

Retrievers inherit Runnable **`batch`** / **`abatch`** — useful for eval and offline benchmarks (**07. Streaming, Batch, and Async**):

In [ ]:
queries = [
    "JWT authentication",
    "Alembic upgrade",
    "pytest fixtures",
]

batch_results = rag_retriever.batch(queries)
for q, docs in zip(queries, batch_results):
    print(f"{q:25s} → {[d.metadata['id'] for d in docs]}")

In [ ]:
import asyncio


async def async_retrieve_demo():
    docs = await rag_retriever.ainvoke("FastAPI routes")
    print("ainvoke:", [d.metadata["id"] for d in docs])


asyncio.run(async_retrieve_demo())

---

## 13. Advanced Retriever Patterns (Conceptual)

LangChain and the ecosystem extend basic vector search:

| Pattern | Idea | When |
|---------|------|------|
| **Multi-query** | LLM rewrites query into variants; merge results | Vague user phrasing |
| **Contextual compression** | Retrieve many, LLM compresses to relevant spans | Long chunks |
| **Parent document** | Search small chunks, return large parent | Precision + context |
| **Ensemble** | Combine BM25 + vector retrievers | Hybrid search |
| **Self-query** | LLM builds metadata filter from natural language | Structured metadata |

These live in integration packages (`langchain`, `langchain-community`) or custom **`BaseRetriever`** subclasses. The **`as_retriever()`** pattern remains the foundation.

In [ ]:
def multi_query_merge(query: str, base: BaseRetriever, rewrites: list[str]) -> list[Document]:
    """Simple multi-query: search original + rewrites, dedupe by id."""
    seen: set[str] = set()
    merged: list[Document] = []
    for q in [query, *rewrites]:
        for doc in base.invoke(q):
            did = doc.metadata.get("id", doc.page_content[:20])
            if did not in seen:
                seen.add(did)
                merged.append(doc)
    return merged


rewrites = ["JWT bearer Authorization header", "API token authentication"]
merged = multi_query_merge("How do I authenticate?", rag_retriever, rewrites)
print("multi-query ids:", [d.metadata["id"] for d in merged])

---

## 14. Logging and Debug Retriever

Wrap retrieval to inspect what the LLM will see:

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def log_retrieval(query: str) -> dict:
    docs = rag_retriever.invoke(query)
    return {
        "question": query,
        "context": format_docs(docs),
        "source_ids": [d.metadata.get("id") for d in docs],
        "num_docs": len(docs),
    }


debug_retrieval = RunnableLambda(log_retrieval)
print(json.dumps(debug_retrieval.invoke("Alembic autogenerate"), indent=2))

---

## 15. Testing with Fake Retrievers

Unit-test RAG prompt assembly without a vector store:

In [ ]:
fake_retriever = RunnableLambda(
    lambda q: [Document(page_content="Fixed context about JWT Bearer tokens.", metadata={"id": "fake"})]
)

test_chain = (
    RunnableParallel(context=fake_retriever | format_docs, question=RunnablePassthrough())
    | rag_prompt
)

prompt_value = test_chain.invoke("any question")
print(prompt_value.messages[0].content[:150], "...")

---

## 16. Retriever Selection Guide

| Scenario | Configuration |
|----------|---------------|
| Default Q&A | `similarity`, `k=3–5` |
| Redundant corpus | `mmr`, `fetch_k=10`, `lambda_mult=0.5` |
| Strict grounding | `similarity_score_threshold` |
| Multi-tenant | `filter={"tenant_id": ...}` |
| Domain routing | Router (**06**) + filtered retriever |
| No vector DB (demo) | `RunnableLambda` keyword retriever |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| `k` too small | Missed relevant chunk | Increase k or use MMR |
| `k` too large | Context overflow / noise | Decrease k; compress |
| Wrong embed model vs index | Bad recall | Same model at index and query |
| Retriever not in RunnableParallel | Lost question key | `question=RunnablePassthrough()` |
| No empty-result handling | LLM hallucinates | `format_docs` fallback + prompt rules |
| Ignoring scores | Irrelevant context | Threshold retriever |
| Sync retriever in tight async loop | Blocked event loop | `ainvoke` / `abatch` |

---

## 18. Summary

**Retrievers** are Runnaables: **`query str → list[Document]`**. **`vectorstore.as_retriever()`** is the primary constructor — configure **`search_type`** (`similarity`, `mmr`, `similarity_score_threshold`) and **`search_kwargs`** (`k`, `filter`, `fetch_k`, `lambda_mult`).

Key takeaways:

- Retrievers bridge vector stores and RAG LCEL chains.
- **`format_docs`** converts documents to prompt-ready strings.
- **MMR** improves diversity; **threshold** rejects weak matches.
- **Custom retrievers** via **`RunnableLambda`** or **`BaseRetriever`**.
- **batch** / **ainvoke** support eval and async servers.

Demonstrations built a vector index, compared similarity vs MMR vs threshold, filtered by metadata, implemented custom retrievers, wired a full RAG preview chain, and showed debug/testing patterns.

Next: **11. RAG with LCEL** — end-to-end RAG, conversational retrieval, and production-oriented chain patterns.